# YOLO11n COCO fine-tune — Colab UI + Local (Mac) Runtime

**Setup**
- Colab is just the front-end. The kernel runs on **your Mac** via *Connect to a local runtime → http://localhost:8888/...*
- Compute units / Colab GPU are irrelevant — your Mac's CPU / Apple-Silicon MPS does the work.
- The `ROBOFLOW_API_KEY` comes from `~/Desktop/object detection /.env`, not Colab Secrets.

**Before running**
1. In a terminal on the Mac:
   ```bash
   cd "~/Desktop/object detection "
   pip install -r requirements.txt jupyter
   caffeinate -i jupyter notebook --NotebookApp.allow_origin='https://colab.research.google.com' --port=8888 --no-browser
   ```
   `caffeinate -i` stops the Mac sleeping during training. Keep this terminal open.
2. In Colab: *Connect → Connect to local runtime →* paste the `?token=...` URL from the jupyter output.
3. *File → Upload notebook →* this file. Run cells top-to-bottom.

**Realistic time on Mac**
- Full COCO @ 30 epochs on M-series MPS ≈ 15–30 h. Don't start blind — run the smoke test below first.
- If that's too slow, drop `EPOCHS` or use the `SMOKE_TEST` switch (≈ 5 min on `coco128`).

In [ ]:
# Smoke test on COCO128 (auto-downloaded, 128 images). Set False for the real run.
SMOKE_TEST = True

WORKSPACE = 'microsoft'
PROJECT   = 'coco'
VERSION   = 50
MODEL     = 'yolo11n.pt'
EPOCHS    = 30        # full run
IMGSZ     = 640
BATCH     = 8         # Mac MPS unified memory is tighter than a T4
RUN_NAME  = 'coco_yolo11n_smoke' if SMOKE_TEST else 'coco_yolo11n'

In [ ]:
import os, platform, subprocess, sys, time
from pathlib import Path

PROJECT_DIR = Path.home() / 'Desktop' / 'object detection '
os.chdir(PROJECT_DIR)
print('Working dir:', os.getcwd())
print('Python    :', sys.version.split()[0])
print('Platform  :', platform.platform())
print('Machine   :', platform.machine())  # 'arm64' on Apple Silicon

In [ ]:
# Install only what the local Mac needs. Ultralytics pulls torch with MPS support automatically.
!pip -q install --upgrade ultralytics roboflow python-dotenv pillow

In [ ]:
from dotenv import load_dotenv
load_dotenv(PROJECT_DIR / '.env')
assert os.environ.get('ROBOFLOW_API_KEY'), 'ROBOFLOW_API_KEY missing in .env'

import torch
if torch.cuda.is_available():
    DEVICE = 0
    print('Device: CUDA', torch.cuda.get_device_name(0))
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    DEVICE = 'mps'
    print('Device: Apple MPS (Metal)')
else:
    DEVICE = 'cpu'
    print('Device: CPU — training will be very slow on full COCO')

In [ ]:
from ultralytics import YOLO

if SMOKE_TEST:
    DATA_YAML = 'coco128.yaml'  # Ultralytics auto-downloads — no Roboflow needed
    EFF_EPOCHS = 3
    print('SMOKE TEST: 3 epochs on coco128 (128 images, ~5 min on MPS).')
else:
    from roboflow import Roboflow
    rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
    dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov8',
                                                                                 location=str(PROJECT_DIR / 'datasets' / f'COCO-{VERSION}'))
    DATA_YAML = str(Path(dataset.location) / 'data.yaml')
    EFF_EPOCHS = EPOCHS
    print('FULL RUN:', EFF_EPOCHS, 'epochs on', DATA_YAML)

In [ ]:
# Resume support: if a previous run left a last.pt, pick up from it.
RUN_DIR = PROJECT_DIR / 'runs' / 'detect' / RUN_NAME
LAST_PT = RUN_DIR / 'weights' / 'last.pt'
start_weights = str(LAST_PT) if LAST_PT.exists() else MODEL
resume = LAST_PT.exists()
print('Starting weights:', start_weights, '| resume:', resume)

model = YOLO(start_weights)
results = model.train(
    data=DATA_YAML,
    epochs=EFF_EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=str(PROJECT_DIR / 'runs' / 'detect'),
    name=RUN_NAME,
    exist_ok=True,
    resume=resume,
    patience=20,
    save=True,
    save_period=1,        # write last.pt every epoch — kill-safe
    plots=True,
    workers=4,            # Mac handles 4 dataloader workers comfortably
    seed=42,
)
BEST = Path(results.save_dir) / 'weights' / 'best.pt'
print('best:', BEST)

In [ ]:
import json
val = model.val(data=DATA_YAML, imgsz=IMGSZ, device=DEVICE, plots=True)
metrics = {
    'mAP50_95': float(val.box.map),
    'mAP50'   : float(val.box.map50),
    'precision': float(val.box.mp),
    'recall'   : float(val.box.mr),
}
(Path(results.save_dir) / 'final_metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

In [ ]:
# Export to every mobile target. INT8 calibrates from the dataset.
# (coremltools doesn't run on Linux easily; on a Mac all three work.)
best_model = YOLO(str(BEST))
out = {}
out['onnx']   = best_model.export(format='onnx',   imgsz=IMGSZ, opset=12, simplify=True)
out['tflite'] = best_model.export(format='tflite', imgsz=IMGSZ, int8=True, data=DATA_YAML, nms=True)
out['coreml'] = best_model.export(format='coreml', imgsz=IMGSZ, int8=True, data=DATA_YAML, nms=True)
for k, v in out.items():
    p = Path(v)
    size = p.stat().st_size / 1e6 if p.exists() else 0
    print(f'{k:7s} -> {v} ({size:.2f} MB)')

In [ ]:
# Copy artefacts to a clean folder for easy hand-off to the Flutter app.
import shutil
ART = PROJECT_DIR / 'weights'
ART.mkdir(exist_ok=True)
shutil.copy2(BEST, ART / 'best.pt')
for k, v in out.items():
    p = Path(v)
    if p.is_dir():
        shutil.copytree(p, ART / p.name, dirs_exist_ok=True)
    elif p.exists():
        shutil.copy2(p, ART / p.name)
print('Artefacts ready in', ART)
!ls -lh '{ART}'

## After training

Run the server pointed at the new weights:
```bash
uvicorn server.main:app --host 0.0.0.0 --port 8000
```

And smoke-test it:
```bash
curl -F image=@samples/test.jpg http://localhost:8000/v1/detect | jq
```

Drop `weights/*_int8.tflite` and `flutter_integration/coco_labels.txt` into the Roshdi Flutter app per `INTEGRATION.md`.